# Credit Card Fraud Detection Using Deep Learning

This notebook compares MLP, 1D CNN, LSTM and Autoencoder models for highly imbalanced credit-card fraud detection. MLP/CNN include BCE, class-weighted BCE and Focal Loss; LSTM uses BCE. A Logistic Regression model is included only as a conventional reference baseline.

**Experiments:** stratified train/validation/test split, duplicate diagnostics, deduplicated rerun, chronological holdout, and five-seed robustness evaluation.

**Important limitations:** the dataset has no customer/card identifiers. CNN/LSTM reshape the 30 tabular features into pseudo-sequences; they do not learn customer transaction histories. V1–V28 are anonymized PCA components. `Time` is elapsed seconds, not a calendar timestamp.

## 1. Install optional explainability packages

In [ ]:
!pip -q install shap lime

## 2. Imports and reproducibility

In [ ]:
import os, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix,
    classification_report, precision_recall_curve)

SEED = 42
EPOCHS = 15
BATCH_SIZE = 512

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 3. Upload and inspect dataset

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_files = [n for n in uploaded if n.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Upload creditcard.csv (CSV file, not ZIP).")
csv_name = next((n for n in csv_files if "creditcard" in n.lower()), csv_files[0])
df = pd.read_csv(csv_name)

required = {"Class", "Time", "Amount"}
if not required.issubset(df.columns):
    raise ValueError(f"Missing required columns: {required - set(df.columns)}")
if df.isnull().sum().sum():
    raise ValueError("Dataset contains missing values; inspect and handle them first.")

print("File:", csv_name, "| Shape:", df.shape)
print("Class counts:\n", df["Class"].value_counts())
print("Fraud percentage:", round(100 * df["Class"].mean(), 4))
display(df.head())

## 4. Define architectures, preprocessing, thresholds and metrics

In [ ]:
def focal_loss(gamma=2.0, alpha=0.75):
    def loss(y_true, y_pred):
        yt = tf.cast(tf.reshape(y_true, (-1, 1)), tf.float32)
        yp = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1-1e-7)
        pt = yt*yp + (1-yt)*(1-yp)
        at = yt*alpha + (1-yt)*(1-alpha)
        return tf.reduce_mean(-at*tf.pow(1-pt, gamma)*tf.math.log(pt))
    return loss

def build_mlp(n):
    return tf.keras.Sequential([
        tf.keras.layers.Input((n,)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(.30),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(.20),
        tf.keras.layers.Dense(1, activation="sigmoid")])

def build_cnn(shape):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape),
        tf.keras.layers.Conv1D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling1D(2),
        tf.keras.layers.Conv1D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(.30),
        tf.keras.layers.Dense(1, activation="sigmoid")])

def build_lstm(shape):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape),
        tf.keras.layers.LSTM(32, dropout=.20),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dropout(.20),
        tf.keras.layers.Dense(1, activation="sigmoid")])

def compile_model(model, loss="binary_crossentropy"):
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss)
    return model

def callbacks():
    return [tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True)]

def choose_threshold(y, scores):
    p, r, t = precision_recall_curve(y, scores)
    if len(t) == 0:
        return 0.5
    p, r = p[:-1], r[:-1]
    f1 = 2*p*r/(p+r+1e-12)
    return float(t[int(np.argmax(f1))])

def metrics_at_threshold(y, scores, threshold):
    pred = (scores >= threshold).astype(int)
    return {
        "Threshold": float(threshold),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "PR-AUC": average_precision_score(y, scores),
        "ROC-AUC": roc_auc_score(y, scores)
    }, pred

def prepare_arrays(train_df, val_df, test_df):
    Xtr = train_df.drop(columns="Class").copy()
    Xv = val_df.drop(columns="Class").copy()
    Xt = test_df.drop(columns="Class").copy()
    ytr = train_df["Class"].to_numpy(dtype=np.int32)
    yv = val_df["Class"].to_numpy(dtype=np.int32)
    yt = test_df["Class"].to_numpy(dtype=np.int32)
    scaler = StandardScaler()
    cols = ["Time", "Amount"]
    Xtr.loc[:, cols] = scaler.fit_transform(Xtr[cols])
    Xv.loc[:, cols] = scaler.transform(Xv[cols])
    Xt.loc[:, cols] = scaler.transform(Xt[cols])
    return (Xtr.to_numpy(np.float32), Xv.to_numpy(np.float32),
            Xt.to_numpy(np.float32), ytr, yv, yt, Xtr.columns.tolist())

def train_autoencoder(Xtr, Xv, ytr, yv, seed):
    set_seed(seed)
    inp = tf.keras.layers.Input(shape=(Xtr.shape[1],))
    z = tf.keras.layers.Dense(16, activation="relu")(inp)
    z = tf.keras.layers.Dense(8, activation="relu")(z)
    z = tf.keras.layers.Dense(16, activation="relu")(z)
    out = tf.keras.layers.Dense(Xtr.shape[1], activation="linear")(z)
    ae = tf.keras.Model(inp, out)
    ae.compile(optimizer="adam", loss="mse")
    ae.fit(Xtr[ytr==0], Xtr[ytr==0],
           validation_data=(Xv[yv==0], Xv[yv==0]),
           epochs=20, batch_size=BATCH_SIZE, callbacks=callbacks(), verbose=0)
    return ae

def reconstruction_errors(model, X):
    rec = model.predict(X, batch_size=BATCH_SIZE, verbose=0)
    return np.mean(np.square(X-rec), axis=1)

## 5. Stratified split and model training pipeline

In [ ]:
def stratified_split(data, seed):
    train, temp = train_test_split(data, test_size=.40, stratify=data["Class"], random_state=seed)
    val, test = train_test_split(temp, test_size=.50, stratify=temp["Class"], random_state=seed)
    return train.copy(), val.copy(), test.copy()

def run_stratified_pipeline(data, seed=42, verbose=0):
    set_seed(seed)
    tr, va, te = stratified_split(data, seed)
    Xtr, Xv, Xt, ytr, yv, yt, feature_names = prepare_arrays(tr, va, te)
    Xtr_seq, Xv_seq, Xt_seq = [x.reshape(-1, x.shape[1], 1) for x in (Xtr, Xv, Xt)]
    cw = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=ytr)
    class_weights = {0: float(cw[0]), 1: float(cw[1])}
    model_specs = {
        "MLP - BCE": (build_mlp(Xtr.shape[1]), "binary_crossentropy", None, Xtr, Xv, Xt),
        "MLP - Weighted BCE": (build_mlp(Xtr.shape[1]), "binary_crossentropy", class_weights, Xtr, Xv, Xt),
        "MLP - Focal Loss": (build_mlp(Xtr.shape[1]), focal_loss(), None, Xtr, Xv, Xt),
        "CNN - BCE": (build_cnn((Xtr.shape[1],1)), "binary_crossentropy", None, Xtr_seq, Xv_seq, Xt_seq),
        "CNN - Weighted BCE": (build_cnn((Xtr.shape[1],1)), "binary_crossentropy", class_weights, Xtr_seq, Xv_seq, Xt_seq),
        "CNN - Focal Loss": (build_cnn((Xtr.shape[1],1)), focal_loss(), None, Xtr_seq, Xv_seq, Xt_seq),
        "LSTM - BCE": (build_lstm((Xtr.shape[1],1)), "binary_crossentropy", None, Xtr_seq, Xv_seq, Xt_seq)
    }
    models, val_scores, test_scores, rows, thresholds = {}, {}, {}, [], {}
    for name, (model, loss, weights, a, b, c) in model_specs.items():
        set_seed(seed)
        compile_model(model, loss)
        fit_args = {}
        if weights is not None:
            fit_args["class_weight"] = weights
        model.fit(a, ytr, validation_data=(b,yv), epochs=EPOCHS,
                  batch_size=BATCH_SIZE, callbacks=callbacks(), verbose=verbose, **fit_args)
        pv = model.predict(b, batch_size=BATCH_SIZE, verbose=0).ravel()
        pt = model.predict(c, batch_size=BATCH_SIZE, verbose=0).ravel()
        th = choose_threshold(yv, pv)
        row, _ = metrics_at_threshold(yt, pt, th)
        row.update({"Model":name, "Validation PR-AUC":average_precision_score(yv,pv)})
        rows.append(row)
        models[name] = model
        val_scores[name], test_scores[name], thresholds[name] = pv, pt, th

    ae = train_autoencoder(Xtr, Xv, ytr, yv, seed)
    av, at = reconstruction_errors(ae, Xv), reconstruction_errors(ae, Xt)
    th = choose_threshold(yv, av)
    row, _ = metrics_at_threshold(yt, at, th)
    row.update({"Model":"Autoencoder", "Validation PR-AUC":average_precision_score(yv,av)})
    rows.append(row)
    models["Autoencoder"] = ae
    val_scores["Autoencoder"], test_scores["Autoencoder"], thresholds["Autoencoder"] = av, at, th

    # Conventional reference baseline, not a deep-learning model.
    lr = LogisticRegression(class_weight="balanced", max_iter=2000, solver="lbfgs", random_state=seed)
    lr.fit(Xtr, ytr)
    lv, lt = lr.predict_proba(Xv)[:,1], lr.predict_proba(Xt)[:,1]
    th = choose_threshold(yv, lv)
    row, _ = metrics_at_threshold(yt, lt, th)
    row.update({"Model":"Logistic Regression (reference baseline)",
                "Validation PR-AUC":average_precision_score(yv,lv)})
    rows.append(row)
    models["Logistic Regression (reference baseline)"] = lr
    val_scores["Logistic Regression (reference baseline)"], test_scores["Logistic Regression (reference baseline)"] = lv, lt
    thresholds["Logistic Regression (reference baseline)"] = th

    table = pd.DataFrame(rows).sort_values("Validation PR-AUC", ascending=False).reset_index(drop=True)
    return {"table":table, "models":models, "val_scores":val_scores,
            "test_scores":test_scores, "thresholds":thresholds,
            "split":(tr,va,te), "arrays":(Xtr,Xv,Xt,ytr,yv,yt,feature_names)}

## 6. Main stratified experiment: original dataset

In [ ]:
original_run = run_stratified_pipeline(df, seed=SEED, verbose=1)
original_results = original_run["table"]
display(original_results)
original_results.to_csv("original_stratified_results.csv", index=False)

best_model_name = original_results.iloc[0]["Model"]
print("Selected using validation PR-AUC:", best_model_name)
best_row = original_results.iloc[0]
print("\nSelected model test metrics (reported once):")
print(best_row[["Precision","Recall","F1","PR-AUC","ROC-AUC"]])

## 7. Duplicate-row diagnostics and deduplicated rerun

In [ ]:
duplicate_count = int(df.duplicated().sum())
tr, va, te = original_run["split"]

def overlap_count(query, reference):
    ref = pd.MultiIndex.from_frame(reference)
    q = pd.MultiIndex.from_frame(query)
    return int(q.isin(ref).sum())

train_features = tr.copy()
val_features = va.copy()
test_features = te.copy()
print("Exact duplicate rows in full original dataset:", duplicate_count)
print("Validation rows with an exact feature duplicate in train:",
      overlap_count(val_features, train_features))
print("Test rows with an exact feature duplicate in train:",
      overlap_count(test_features, train_features))

df_dedup = df.drop_duplicates().reset_index(drop=True)
print("Rows before:", len(df), "| after deduplication:", len(df_dedup),
      "| removed:", len(df)-len(df_dedup))
dedup_run = run_stratified_pipeline(df_dedup, seed=SEED, verbose=0)
dedup_results = dedup_run["table"]

comparison = pd.concat([
    original_results.assign(Dataset="Original"),
    dedup_results.assign(Dataset="Deduplicated")
], ignore_index=True)
comparison.to_csv("original_vs_deduplicated_results.csv", index=False)
display(comparison[["Dataset","Model","Precision","Recall","F1","PR-AUC","ROC-AUC"]]
        .sort_values(["Model","Dataset"]))

## 8. Five-seed evaluation on the original stratified split

In [ ]:
def run_seed_metrics(data, seed):
    run = run_stratified_pipeline(data, seed=seed, verbose=0)
    return run["table"][["Model","PR-AUC","Recall","Precision","F1"]]

seed_values = [42, 1, 2, 3, 4]
seed_frames = []
for s in seed_values:
    print("Running seed:", s)
    tmp = run_seed_metrics(df, s)
    tmp["Seed"] = s
    seed_frames.append(tmp)

seed_results = pd.concat(seed_frames, ignore_index=True)
summary = seed_results.groupby("Model")[["PR-AUC","Recall","Precision","F1"]].agg(["mean","std"])
summary.columns = [f"{a}_{b}" for a,b in summary.columns]
summary = summary.reset_index()
display(summary)
summary.to_csv("five_seed_mean_std.csv", index=False)
seed_results.to_csv("five_seed_raw_results.csv", index=False)

## 9. Chronological holdout experiment

In [ ]:
def chronological_split(data):
    ordered = data.sort_values("Time", kind="mergesort").reset_index(drop=True)
    n = len(ordered)
    return (ordered.iloc[:int(.60*n)].copy(),
            ordered.iloc[int(.60*n):int(.80*n)].copy(),
            ordered.iloc[int(.80*n):].copy())

def run_chronological(data, seed=42):
    set_seed(seed)
    tr, va, te = chronological_split(data)
    print("Chronological split fraud counts:",
          {"train":int(tr.Class.sum()),"validation":int(va.Class.sum()),"test":int(te.Class.sum())})
    Xtr,Xv,Xt,ytr,yv,yt,features = prepare_arrays(tr,va,te)
    seq = [x.reshape(-1,x.shape[1],1) for x in (Xtr,Xv,Xt)]
    specs = {
        "MLP - BCE":(build_mlp(Xtr.shape[1]),"binary_crossentropy",Xtr,Xv,Xt),
        "MLP - Focal Loss":(build_mlp(Xtr.shape[1]),focal_loss(),Xtr,Xv,Xt),
        "CNN - BCE":(build_cnn((Xtr.shape[1],1)),"binary_crossentropy",seq[0],seq[1],seq[2]),
        "LSTM - BCE":(build_lstm((Xtr.shape[1],1)),"binary_crossentropy",seq[0],seq[1],seq[2])
    }
    rows=[]
    for name,(model,loss,a,b,c) in specs.items():
        set_seed(seed)
        compile_model(model,loss)
        model.fit(a,ytr,validation_data=(b,yv),epochs=EPOCHS,batch_size=BATCH_SIZE,
                  callbacks=callbacks(),verbose=0)
        pv=model.predict(b,batch_size=BATCH_SIZE,verbose=0).ravel()
        pt=model.predict(c,batch_size=BATCH_SIZE,verbose=0).ravel()
        threshold=choose_threshold(yv,pv)
        row,_=metrics_at_threshold(yt,pt,threshold)
        row.update({"Model":name,"Validation PR-AUC":average_precision_score(yv,pv)})
        rows.append(row)

    ae=train_autoencoder(Xtr,Xv,ytr,yv,seed)
    av,at=reconstruction_errors(ae,Xv),reconstruction_errors(ae,Xt)
    threshold=choose_threshold(yv,av)
    row,_=metrics_at_threshold(yt,at,threshold)
    row.update({"Model":"Autoencoder","Validation PR-AUC":average_precision_score(yv,av)})
    rows.append(row)
    return pd.DataFrame(rows).sort_values("Validation PR-AUC",ascending=False).reset_index(drop=True)

chronological_results = run_chronological(df, SEED)
display(chronological_results)
chronological_results.to_csv("chronological_results.csv",index=False)

## 10. Confusion matrix and training-independent model comparison

In [ ]:
# Detailed test report for the validation-PR-AUC-selected model only.
best_name = original_results.iloc[0]["Model"]
_, _, y_test = original_run["arrays"][3:6]
best_scores = original_run["test_scores"][best_name]
best_threshold = original_run["thresholds"][best_name]
best_metrics, best_pred = metrics_at_threshold(y_test,best_scores,best_threshold)
print("Best model:",best_name)
print("Threshold selected on validation:",round(best_threshold,6))
print("\n",classification_report(y_test,best_pred,digits=4,zero_division=0))
print("Confusion matrix:\n",confusion_matrix(y_test,best_pred))

plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test,best_pred),annot=True,fmt="d",cmap="Blues")
plt.title("Selected model confusion matrix: "+best_name)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

original_results.set_index("Model")[["Validation PR-AUC","PR-AUC","Recall","Precision","F1"]].plot(
    kind="bar",figsize=(13,5),ylim=(0,1),rot=35)
plt.title("Stratified experiment: validation ranking and test metrics")
plt.tight_layout(); plt.show()

## 11. SHAP explanation: all test frauds plus random genuine transactions

In [ ]:
import shap

Xtr,Xv,Xt,ytr,yv,yt,feature_names = original_run["arrays"]
best_mlp_variants = {
    "MLP - BCE": original_run["models"]["MLP - BCE"],
    "MLP - Weighted BCE": original_run["models"]["MLP - Weighted BCE"],
    "MLP - Focal Loss": original_run["models"]["MLP - Focal Loss"]
}
# Select the MLP variant by validation PR-AUC only.
best_mlp_name = max(best_mlp_variants,
    key=lambda n: average_precision_score(yv,
        best_mlp_variants[n].predict(Xv,batch_size=BATCH_SIZE,verbose=0).ravel()))
best_mlp = best_mlp_variants[best_mlp_name]
fraud_idx = np.where(yt==1)[0]
genuine_idx = np.where(yt==0)[0]
rng = np.random.default_rng(SEED)
genuine_sample = rng.choice(genuine_idx, size=min(50,len(genuine_idx)), replace=False)
explain_idx = np.concatenate([fraud_idx,genuine_sample])
explain_data = Xt[explain_idx]
background_idx = rng.choice(len(Xtr), size=min(40,len(Xtr)), replace=False)
background = Xtr[background_idx]
print("SHAP model:",best_mlp_name,"| samples:",len(explain_data),
      "| fraud:",len(fraud_idx),"| genuine:",len(genuine_sample))
explainer = shap.KernelExplainer(
    lambda a: best_mlp.predict(np.asarray(a,dtype=np.float32),batch_size=BATCH_SIZE,verbose=0).ravel(),
    background)
shap_values = explainer.shap_values(explain_data,nsamples=500)
sv = shap_values[0] if isinstance(shap_values,list) else shap_values
shap.summary_plot(sv,explain_data,feature_names=feature_names)

## 12. LIME explanations: one true positive and one false negative

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    Xtr,feature_names=feature_names,class_names=["Genuine","Fraud"],mode="classification")
def predict_classes(a):
    p=best_mlp.predict(np.asarray(a,dtype=np.float32),batch_size=BATCH_SIZE,verbose=0).ravel()
    return np.column_stack([1-p,p])

mlp_test_scores = best_mlp.predict(Xt,batch_size=BATCH_SIZE,verbose=0).ravel()
mlp_threshold = choose_threshold(yv,best_mlp.predict(Xv,batch_size=BATCH_SIZE,verbose=0).ravel())
mlp_pred = (mlp_test_scores>=mlp_threshold).astype(int)
tp_idx = np.where((yt==1)&(mlp_pred==1))[0]
fn_idx = np.where((yt==1)&(mlp_pred==0))[0]
if len(tp_idx):
    print("LIME: true positive")
    lime_explainer.explain_instance(Xt[tp_idx[0]],predict_classes,
        num_features=10,num_samples=1000).show_in_notebook(show_table=True)
else:
    print("No true positive found at the selected threshold.")
if len(fn_idx):
    print("LIME: false negative")
    lime_explainer.explain_instance(Xt[fn_idx[0]],predict_classes,
        num_features=10,num_samples=1000).show_in_notebook(show_table=True)
else:
    print("No false negative found at the selected threshold.")

## 13. Save final tables and trained models

In [ ]:
original_results.to_csv("model_comparison_results.csv",index=False)
comparison.to_csv("original_vs_deduplicated_results.csv",index=False)
chronological_results.to_csv("chronological_results.csv",index=False)
summary.to_csv("five_seed_mean_std.csv",index=False)
seed_results.to_csv("five_seed_raw_results.csv",index=False)

# Save models from the main original stratified run.
for name,model in original_run["models"].items():
    filename = name.lower().replace(" ","_").replace("-","").replace("(","").replace(")","") + ".keras"
    if isinstance(model, tf.keras.Model):
        model.save(filename)

print("Saved result tables and neural-network models.")

## 14. Download result tables

In [ ]:
from google.colab import files
for filename in ["model_comparison_results.csv","original_vs_deduplicated_results.csv",
                 "chronological_results.csv","five_seed_mean_std.csv","five_seed_raw_results.csv"]:
    files.download(filename)

## 15. Results Summary, Limitations and Conclusion

### Results Summary
The notebook selects the main model using **validation PR-AUC**, then reports that selected model's held-out test metrics. Review the generated CSV files for:
- Original stratified model comparison
- Original versus deduplicated comparison
- Chronological holdout results
- Five-seed mean and standard deviation

### Limitations
- The stratified test set contains only about 99 fraud samples, so small metric differences are not necessarily statistically significant.
- CNN and LSTM use pseudo-sequences formed from anonymized PCA features; these are not real customer transaction sequences.
- `Time` is elapsed seconds, not a calendar timestamp.
- No external dataset validation has been performed.
- The Logistic Regression model is a non-deep-learning reference baseline.
- Autoencoder reconstruction error is an anomaly score, not a calibrated fraud probability.
- SHAP and LIME explain model outputs; they do not establish causality.
- Real deployment would require external validation, cost-sensitive threshold selection, calibration, monitoring, and security/privacy review.

### Conclusion
Use the model ranked first by validation PR-AUC as the primary candidate, while considering its held-out precision, recall and F1. Do not assume the highest test F1 alone makes a model best. If MLP-BCE remains top-ranked, it is a defensible recommendation because it provides the strongest validation ranking performance in this experiment and a strong balance of fraud precision and recall. Revisit this recommendation if the new run selects a different model.